# 698. Partition to K Equal Sum Subsets

## Topic Alignment
- State compression DP with bitmasks is essential for ML workflow optimization problems like model ensemble selection, feature subset selection, resource allocation across workers, and batch scheduling.
- The pattern of tracking "which items are used" appears in hyperparameter optimization, A/B test bucket assignment, and distributed training job scheduling.
- Bit manipulation for state representation is a crucial optimization technique for memory-constrained systems.

## Metadata 摘要
- Source: https://leetcode.com/problems/partition-to-k-equal-sum-subsets/
- Tags: Dynamic Programming, Backtracking, Bit Manipulation, State Compression, Memoization
- Difficulty: Medium
- Priority: High

## Problem Statement 原题描述
Given an integer array `nums` and an integer `k`, return `true` if it is possible to divide this array into `k` non-empty subsets whose sums are all equal.

**Constraints**:
- `1 <= k <= nums.length <= 16`
- `1 <= nums[i] <= 10^4`
- The frequency of each element is in the range `[1, 4]`.

## Progressive Hints
- Hint 1: If total sum is not divisible by k, return false immediately.
- Hint 2: Each subset must have sum = total_sum / k (call it target).
- Hint 3: Use bitmask to represent which numbers have been used (max 16 bits for n <= 16).
- Hint 4: State: `dp[mask]` = can we partition the used numbers (represented by mask) into valid complete subsets?
- Hint 5: For each state, try adding unused numbers to the current incomplete subset.
- Hint 6: When current subset reaches target sum, start a new subset.
- Hint 7: Sort nums in descending order to prune search space early.
- Hint 8: Memoize based on (mask, current_sum % target) to avoid recomputation.

## Solution Overview
Use **state compression DP with bitmask** + **backtracking with memoization**.

**Key Observations**:
1. Target sum per subset = `total_sum / k`
2. Need to partition n numbers into k subsets (n <= 16 → bitmask feasible)
3. State: which numbers are used (bitmask)

**State Definition**:
- `mask`: bitmask representing which numbers are used
- `current_sum`: sum of current incomplete subset being built
- `dp[(mask, current_sum % target)]`: boolean, can we achieve valid partition from this state?

**Why modulo target?**
- When `current_sum == target`, we complete one subset and reset to 0
- Only `current_sum % target` matters for state uniqueness
- This reduces state space significantly

**Recurrence**:
```python
For each unused number i:
    if current_sum + nums[i] <= target:
        new_mask = mask | (1 << i)
        new_sum = (current_sum + nums[i]) % target
        if dfs(new_mask, new_sum): return True
return False
```

**Optimizations**:
1. Sort descending: larger numbers first, faster pruning
2. Early termination: if any number > target, return False
3. Memoization: cache (mask, current_sum % target)

## Detailed Explanation

### Understanding State Compression

**Bitmask basics**:
- `mask = 0b1011` means indices 0, 1, 3 are used
- `mask & (1 << i)`: check if index i is used
- `mask | (1 << i)`: mark index i as used
- `mask ^ (1 << i)`: toggle index i

**Why use bitmask?**
- Compact representation: n <= 16 fits in one integer
- Fast operations: bitwise ops are O(1)
- Easy memoization: integer can be hash key

---

### Problem Analysis

**Goal**: Partition `nums` into k subsets, each with sum = `total_sum / k`

**Constraints**:
- Each number used exactly once
- All k subsets must have equal sum
- Subsets can be different sizes (only sum matters)

**Why DP + Backtracking?**
- Pure backtracking: O(k^n) - try each number in each subset
- With memoization: avoid recomputing same states
- Bitmask DP: state space = 2^n × target (manageable for n <= 16)

---

### State Definition Deep Dive

**State components**:
1. `mask`: which numbers are used (complete + partial subsets)
2. `current_sum`: sum of current incomplete subset

**Why track current_sum?**
- We build subsets one at a time
- Need to know when current subset is complete (sum == target)
- When complete, start new subset (reset sum to 0)

**Example state**:
```
nums = [4, 3, 2, 3, 5, 2, 1], k = 4, target = 5
mask = 0b0001101 (used indices 0, 2, 3)
current_sum = 4 (nums[0]=4, partial)
```

**State transition**:
- Try adding nums[1] = 3: current_sum + 3 = 7 > 5 ✗ (skip)
- Try adding nums[6] = 1: current_sum + 1 = 5 ✓ (complete subset, reset to 0)
- New state: mask = 0b1001101, current_sum = 0

---

### Why Modulo Target?

**Key insight**: When current_sum reaches target, it resets to 0

**States that are equivalent**:
- `(mask, current_sum=5)` when target=5 → about to complete subset
- `(mask | (1<<i), current_sum=0)` after adding last number → completed subset

**Only remainder matters**:
- `current_sum = 0` and `current_sum = target` are both "starting new subset"
- `current_sum = 2` and `current_sum = target+2` are same progress
- Use `current_sum % target` as state component

**State space reduction**:
- Without modulo: 2^n × 10000 states (current_sum can be large)
- With modulo: 2^n × target states (much smaller)
- For n=16, target=20: 65536 × 20 = 1.3M states (feasible)

---

### Bit Manipulation Techniques

**Check if index i is used**:
```python
if mask & (1 << i):
    # index i is used
```
- `1 << i`: create mask with only bit i set
- `mask & (1 << i)`: AND operation, non-zero if bit i is set

**Mark index i as used**:
```python
new_mask = mask | (1 << i)
```
- `mask | (1 << i)`: OR operation, sets bit i to 1

**Count number of bits set**:
```python
count = bin(mask).count('1')
# Or use Brian Kernighan's algorithm
count = 0
while mask:
    mask &= mask - 1  # Remove lowest set bit
    count += 1
```

**Iterate over all subsets of mask**:
```python
subset = mask
while subset:
    # process subset
    subset = (subset - 1) & mask
```

---

### Algorithm Walkthrough

**Input**: `nums = [4, 3, 2, 3, 5, 2, 1]`, `k = 4`

**Step 1: Validation**
- total_sum = 20
- 20 % 4 = 0 ✓
- target = 20 / 4 = 5
- max(nums) = 5 <= 5 ✓

**Step 2: Sort descending**
- nums = [5, 4, 3, 3, 2, 2, 1]
- Larger numbers first helps prune faster

**Step 3: DFS with memoization**
- Start: mask = 0b0000000, current_sum = 0
- Try nums[0]=5: mask = 0b0000001, sum = 5 (complete subset!)
  - Reset sum to 0, continue
- Try nums[1]=4: mask = 0b0000011, sum = 4
  - Try nums[6]=1: mask = 0b1000011, sum = 5 (complete!)
- Continue until all numbers used or impossible

**Step 4: Memoization**
- Cache results for (mask, sum % target)
- Avoid recomputing same states

---

### Optimization: Sorting Descending

**Why sort large to small?**
- Large numbers are harder to fit
- If a large number can't fit anywhere, fail early
- Reduces branching factor in search tree

**Example**:
- `nums = [1, 1, 1, 1, 10]`, `target = 11`
- If we try 1's first: many combinations before discovering 10 must go somewhere
- If we try 10 first: immediately know it needs exactly one 1, prune other options

---

### Time Complexity Analysis

**State space**: O(2^n × target)
- n subsets → 2^n possible masks
- current_sum ∈ [0, target-1] → target possibilities

**Transitions per state**: O(n)
- Try each of n numbers

**Total**: O(2^n × target × n)
- For n=16: 2^16 × target × 16 = 1M × target operations
- With target ~ 1000, still manageable

**Without memoization**: O(k^n) - exponentially worse

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Backtracking only | O(k^n) | O(n) | Exponential, no memoization |
| DP + bitmask | O(2^n × target × n) | O(2^n × target) | Optimal for small n |
| Subset sum DP | O(n × sum^k) | O(sum^k) | Pseudo-polynomial |
| Greedy | O(n log n) | O(1) | Incorrect, doesn't work |

In [ ]:
from typing import List

class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        """
        State compression DP with bitmask + memoization.
        
        Time: O(2^n × target × n) where n = len(nums)
        Space: O(2^n × target) for memoization
        """
        total = sum(nums)
        
        # Quick validation
        if total % k != 0:
            return False
        
        target = total // k
        n = len(nums)
        
        # If any number exceeds target, impossible
        if max(nums) > target:
            return False
        
        # Sort descending for better pruning
        nums.sort(reverse=True)
        
        # Memoization: (mask, current_sum % target) -> bool
        memo = {}
        
        def dfs(mask, current_sum):
            """
            Returns True if we can partition remaining numbers into valid subsets.
            
            mask: bitmask of used numbers
            current_sum: sum of current incomplete subset
            """
            # All numbers used successfully
            if mask == (1 << n) - 1:
                return True
            
            # Check memo (use modulo for state compression)
            state = (mask, current_sum % target)
            if state in memo:
                return memo[state]
            
            # Try adding each unused number
            for i in range(n):
                # Skip if already used
                if mask & (1 << i):
                    continue
                
                # Skip if adding this number exceeds target
                if current_sum + nums[i] > target:
                    continue
                
                # Mark this number as used
                new_mask = mask | (1 << i)
                new_sum = current_sum + nums[i]
                
                # If we complete a subset, reset sum to 0
                if new_sum == target:
                    new_sum = 0
                
                # Recursively try to partition remaining numbers
                if dfs(new_mask, new_sum):
                    memo[state] = True
                    return True
            
            memo[state] = False
            return False
        
        return dfs(0, 0)

In [ ]:
# Test cases
tests = [
    ([4,3,2,3,5,2,1], 4, True),    # Can partition into [5], [4,1], [3,2], [3,2]
    ([1,2,3,4], 3, False),          # Cannot partition equally
    ([1,1,1,1,1,1,1,1,1,1], 5, True),  # [1,1], [1,1], [1,1], [1,1], [1,1]
    ([2,2,2,2,3,4,5], 4, False),   # sum=20, target=5, but can't partition
    ([1,1,1,1,2,2,2,2], 4, True),  # Multiple valid partitions
]

solver = Solution()
for nums, k, expected in tests:
    result = solver.canPartitionKSubsets(nums, k)
    assert result == expected, f"Failed for nums={nums}, k={k}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(2^n × target × n) where n = len(nums)
  - State space: 2^n masks × target possible sums
  - Each state tries n numbers
  - With memoization, each state computed once
- **Space**: O(2^n × target) for memoization dictionary
  - Plus O(n) recursion stack
  - Feasible for n <= 16

## Edge Cases & Pitfalls
- **Total not divisible by k**: Return False immediately
- **Any number > target**: Impossible to fit in any subset
- **k = 1**: Always True (entire array is one subset)
- **k = n**: True only if all numbers are equal
- **All numbers equal**: Always True if sum divisible by k
- **Duplicate numbers**: Algorithm handles correctly with bitmask (positional, not value-based)
- **Common mistake**: Forgetting to reset sum to 0 when subset completes
- **Common mistake**: Not using modulo in memo key (state space explosion)
- **Common mistake**: Iterating from small to large (sorting descending is crucial)
- **Optimization**: Can use `(1 << n) - 1` as all-ones mask

## Follow-up Variants
- **Minimize k**: Find minimum k for valid partition (binary search on k)
- **Count solutions**: How many ways to partition (remove early return)
- **Weighted partition**: Numbers have weights, partition by weight sum
- **Multiple constraints**: Each subset must satisfy size AND sum constraints
- **Online version**: Numbers arrive one by one, assign to subsets dynamically
- **Approximate**: Allow epsilon difference in subset sums
- **Maximize min**: Maximize minimum subset sum (different objective)
- **2D partition**: Partition matrix into k equal-sum rectangular regions

## Takeaways
- **State compression with bitmask** is powerful for small n (typically n <= 20).
- **Bit manipulation** provides fast, compact state representation.
- Using **modulo in state** can dramatically reduce state space.
- **Sorting descending** helps prune search space in backtracking.
- Combining **DP + backtracking** leverages strengths of both approaches.
- Memoization key design is crucial: include only what affects future decisions.
- Understanding when current_sum == target → reset to 0 is key insight.
- This pattern extends to many subset selection and partition problems.
- **State compression DP** trades space for time, feasible when state space is manageable.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 416 | Partition Equal Subset Sum | DP, k=2 special case |
| LC 473 | Matchsticks to Square | State compression DP, k=4 |
| LC 1986 | Minimum Number of Work Sessions to Finish the Tasks | Bitmask DP |
| LC 847 | Shortest Path Visiting All Nodes | Bitmask DP on graphs |
| LC 1723 | Find Minimum Time to Finish All Jobs | Backtracking + pruning |
| LC 2305 | Fair Distribution of Cookies | Similar partition problem |